# 🏥 RAG-Type Architectural Compliance Checker

**Pipeline overview:**
```
Manifest JSON  →  Structured Data Extraction
                          ↓
            Deterministic Rules Engine  →  PASS/FAIL results
                          ↓
            Gemini LLM  →  Human-readable report
```

**What this checks (based on NBC 2016 / NABH):**
- Room minimum areas by type
- Corridor minimum width
- Toilet minimum dimensions
- Balcony minimum width
- Staircase clear width
- Kitchen minimum area
- Bedroom minimum area and width
- Adjacency: toilet must be adjacent to bedroom it serves
- Adjacency: kitchen must be adjacent to living/dining
- Fire safety: staircase must be adjacent to fire tower / lobby

## Step 1: Install dependencies

In [67]:
!pip install -q google-genai

## Step 2: Imports and configuration

In [83]:
import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from google import genai
from google.colab import userdata

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
MANIFEST_PATH   = "manifest_TOWER-B_GF.json"
GEMINI_MODEL    = "gemini-3.5-flash-lite"

# ─── NBC 2016 / NABH THRESHOLDS (all in m² or mm) ─────────────────────────────
RULES = {
    # Room type keyword  →  (min_area_m2, min_width_mm)
    "M.BEDROOM":          (9.5,  2700),
    "CHILDREN BEDRM":     (7.5,  2400),
    "GUEST BEDRM":        (7.5,  2400),
    "KITCHEN":            (5.0,  1800),
    "TOILET":             (2.0,  1200),
    "WC":                 (1.1,  900),
    "BATH":               (1.1,  900),
    "BALCONY":            (None, 1200),
    "UTILITY":            (2.0,  None),
    "RECEPTION":          (15.0, None),
    "MULTIPURPOSE HALL":  (20.0, None),
    "LOBBY":              (5.0,  None),
    "ELECTRICAL ROOM":    (10.0, None),
}

MIN_CORRIDOR_WIDTH_MM   = 1800
MIN_STAIRCASE_WIDTH_MM  = 1000

ADJACENCY_RULES = [
    {
        "rule_name":      "Bedroom must have accessible toilet",
        "source_types":   ["M.BEDROOM", "CHILDREN BEDRM", "GUEST BEDRM"],
        "must_be_near":   ["TOILET", "BATH", "WC"],
        "description":    "Every bedroom must be adjacent to a toilet/bath",
    },
    {
        "rule_name":      "Kitchen must connect to living/dining",
        "source_types":   ["KITCHEN"],
        "must_be_near":   ["DINING", "LIVING", "FOYER"],
        "description":    "Kitchen must be directly accessible from dining or living area",
    },
    {
        "rule_name":      "Staircase must be near fire exit / lobby",
        "source_types":   ["STAIRCASE"],
        "must_be_near":   ["FIRE TOWER", "LOBBY", "CORRIDOR", "FIRE LIFT"],
        "description":    "Staircase must be adjacent to a fire exit, fire tower or lobby",
    },
    {
        "rule_name":      "Reception must connect to corridor or lobby",
        "source_types":   ["RECEPTION"],
        "must_be_near":   ["CORRIDOR", "LOBBY", "FOYER"],
        "description":    "Reception must be accessible from a corridor or lobby",
    },
]

print("✅ Configuration loaded.")

✅ Configuration loaded.


## Step 3: Load and parse the manifest JSON

In [69]:
@dataclass
class Room:
    """
    A clean, flat representation of one room extracted from the manifest.
    This is what gets fed to both the rules engine and the LLM.
    """
    id:             str
    raw_label:      str
    labels:         list[str]          # can be multiple e.g. [DINING, LIVING, FOYER]
    status:         str                # room / unlabelled / corridor
    is_corridor:    bool
    area_m2:        float
    width_mm:       int                # inscribed_width_mm — the usable clear width
    mrr_dims_mm:    list[int]          # [shorter_dim, longer_dim]
    adjacent:       list[str]          # list of adjacent room IDs
    doors:          list[str]
    windows:        list[str]
    confidence:     float
    notes:          list[str]


def load_manifest(path: str) -> tuple[list[Room], dict]:
    """
    Load the manifest JSON and convert every room entry into a Room dataclass.
    Also returns the raw dict for reference.
    """
    with open(path) as f:
        data = json.load(f)

    rooms = []
    for r in data.get("rooms", []):
        room = Room(
            id           = r["id"],
            raw_label    = r.get("raw_label") or "UNLABELLED",
            labels       = r.get("labels", []),
            status       = r.get("status", "room"),
            is_corridor  = r.get("is_corridor", False),
            area_m2      = r.get("area_m2", 0.0),
            width_mm     = r.get("inscribed_width_mm") or r.get("clear_width_mm") or 0,
            mrr_dims_mm  = r.get("mrr_dims_mm", [0, 0]),
            adjacent     = r.get("adjacent", []),
            doors        = r.get("doors", []),
            windows      = r.get("windows", []),
            confidence   = r.get("confidence", 1.0),
            notes        = r.get("notes", []),
        )
        rooms.append(room)

    print(f"✅ Loaded {len(rooms)} rooms from manifest.")
    print(f"   Source file: {data.get('source_file', 'unknown')}")
    print(f"   Floor: {data.get('floor', {}).get('block', 'unknown')}")
    return rooms, data


rooms, raw_data = load_manifest('/content/manifest_TOWER-B_GF.json')

✅ Loaded 200 rooms from manifest.
   Source file: TOWER-B GF(1).dxf
   Floor: GROUND FLOOR PLAN 28-09-2021


## Step 4: Build an index for fast adjacency lookup

In [70]:
def build_room_index(rooms: list[Room]) -> dict[str, Room]:
    """Build a dict: room_id → Room object for O(1) lookups."""
    return {r.id: r for r in rooms}


def room_matches_type(room: Room, type_keywords: list[str]) -> bool:
    """
    Check if a room's labels or raw_label contain any of the given keywords.
    Case-insensitive.

    Example:
        room.labels = ["DINING", "LIVING", "FOYER"]
        room_matches_type(room, ["DINING"]) → True
    """
    all_text = " ".join(room.labels + [room.raw_label]).upper()
    return any(kw.upper() in all_text for kw in type_keywords)


room_index = build_room_index(rooms)
print(f"✅ Room index built: {len(room_index)} entries.")

✅ Room index built: 200 entries.


## Step 5: Deterministic compliance engine

This is the core of the pipeline. Every check returns a structured result dict.
**The LLM does NOT do these checks — Python does.**

In [71]:
@dataclass
class CheckResult:
    rule_name:   str
    room_id:     str
    room_label:  str
    status:      str           # PASS / FAIL / SKIP
    measured:    str           # what was actually measured
    required:    str           # what the regulation requires
    detail:      str           # human-readable explanation


def check_room_area(room: Room, min_area: float) -> CheckResult:
    """Check if room area meets the minimum."""
    status = "PASS" if room.area_m2 >= min_area else "FAIL"
    return CheckResult(
        rule_name  = "Minimum room area",
        room_id    = room.id,
        room_label = room.raw_label,
        status     = status,
        measured   = f"{room.area_m2:.2f} m²",
        required   = f">= {min_area} m²",
        detail     = f"{room.raw_label} ({room.id}): area is {room.area_m2:.2f} m², "
                     f"minimum is {min_area} m²",
    )


def check_room_width(room: Room, min_width_mm: int) -> CheckResult:
    """Check if room's inscribed (usable) width meets the minimum."""
    status = "PASS" if room.width_mm >= min_width_mm else "FAIL"
    return CheckResult(
        rule_name  = "Minimum room width",
        room_id    = room.id,
        room_label = room.raw_label,
        status     = status,
        measured   = f"{room.width_mm} mm",
        required   = f">= {min_width_mm} mm",
        detail     = f"{room.raw_label} ({room.id}): width is {room.width_mm} mm, "
                     f"minimum is {min_width_mm} mm",
    )


def check_corridor_width(room: Room) -> CheckResult:
    """Check corridor meets minimum width."""
    w = room.mrr_dims_mm[0] if room.mrr_dims_mm else room.width_mm
    status = "PASS" if w >= MIN_CORRIDOR_WIDTH_MM else "FAIL"
    return CheckResult(
        rule_name  = "Corridor minimum width",
        room_id    = room.id,
        room_label = room.raw_label,
        status     = status,
        measured   = f"{w} mm",
        required   = f">= {MIN_CORRIDOR_WIDTH_MM} mm",
        detail     = f"Corridor {room.id}: measured width {w} mm, minimum {MIN_CORRIDOR_WIDTH_MM} mm",
    )


def check_staircase_width(room: Room) -> CheckResult:
    """Check staircase clear width (uses inscribed_width which reflects tread width)."""
    # For staircases, inscribed_width_mm represents the clear flight width
    w = room.width_mm
    status = "PASS" if w >= MIN_STAIRCASE_WIDTH_MM else "FAIL"
    return CheckResult(
        rule_name  = "Staircase minimum clear width",
        room_id    = room.id,
        room_label = room.raw_label,
        status     = status,
        measured   = f"{w} mm",
        required   = f">= {MIN_STAIRCASE_WIDTH_MM} mm",
        detail     = f"Staircase {room.id}: clear width {w} mm, minimum {MIN_STAIRCASE_WIDTH_MM} mm",
    )


def check_adjacency(room: Room, must_be_near: list[str],
                    rule_name: str, room_index: dict) -> CheckResult:
    """
    Check that at least one of this room's adjacent rooms matches one of
    the 'must_be_near' keywords.

    HOW IT WORKS:
    - The manifest stores each room's 'adjacent' list as room IDs
    - We look up each adjacent room and check if its label matches
      any of the required keywords
    """
    found_adjacent = []
    for adj_id in room.adjacent:
        if adj_id == "EXTERIOR":
            continue
        adj_room = room_index.get(adj_id)
        if adj_room and room_matches_type(adj_room, must_be_near):
            found_adjacent.append(f"{adj_room.raw_label} ({adj_id})")

    status = "PASS" if found_adjacent else "FAIL"
    return CheckResult(
        rule_name  = rule_name,
        room_id    = room.id,
        room_label = room.raw_label,
        status     = status,
        measured   = ", ".join(found_adjacent) if found_adjacent else "none found",
        required   = f"adjacent to one of: {', '.join(must_be_near)}",
        detail     = (
            f"{room.raw_label} ({room.id}) is adjacent to: {', '.join(found_adjacent)}"
            if found_adjacent
            else f"{room.raw_label} ({room.id}) has no adjacent {'/'.join(must_be_near)}"
        ),
    )


print("✅ Compliance functions defined.")

✅ Compliance functions defined.


## Step 6: Run all checks

In [72]:
def run_all_checks(rooms: list[Room], room_index: dict) -> list[CheckResult]:
    """
    Loop through every room and run all applicable checks.

    Logic:
    - If room matches a type in RULES, run area and width checks for that type
    - If room is a corridor (is_corridor=True), run corridor width check
    - If room label contains STAIRCASE, run staircase width check
    - For each ADJACENCY_RULES entry, find matching rooms and check adjacency
    """
    results = []

    for room in rooms:
        # Skip unlabelled rooms and very low-confidence detections
        if room.status == "unlabelled" or room.confidence < 0.7:
            continue

        # ── 1. Area and width checks by room type ──────────────────────────────
        for type_keyword, (min_area, min_width) in RULES.items():
            if room_matches_type(room, [type_keyword]):
                if min_area is not None:
                    results.append(check_room_area(room, min_area))
                if min_width is not None:
                    results.append(check_room_width(room, min_width))
                break  # stop after first matching rule (most specific match)

        # ── 2. Corridor width check ────────────────────────────────────────────
        if room.is_corridor:
            results.append(check_corridor_width(room))

        # ── 3. Staircase width check ────────────────────────────────────────────
        if room_matches_type(room, ["STAIRCASE"]):
            results.append(check_staircase_width(room))

    # ── 4. Adjacency checks ────────────────────────────────────────────────────
    for adj_rule in ADJACENCY_RULES:
        for room in rooms:
            if room.status == "unlabelled" or room.confidence < 0.7:
                continue
            if room_matches_type(room, adj_rule["source_types"]):
                results.append(
                    check_adjacency(
                        room,
                        adj_rule["must_be_near"],
                        adj_rule["rule_name"],
                        room_index,
                    )
                )

    return results


all_results = run_all_checks(rooms, room_index)

passed = sum(1 for r in all_results if r.status == "PASS")
failed = sum(1 for r in all_results if r.status == "FAIL")
print(f"✅ Checks complete: {len(all_results)} total | {passed} PASS | {failed} FAIL")

✅ Checks complete: 266 total | 255 PASS | 11 FAIL


## Step 7: Print deterministic PASS/FAIL table

In [73]:
def print_compliance_table(results: list[CheckResult], show_passes: bool = False):
    """
    Print a clean PASS/FAIL summary table.

    By default shows only failures to keep output compact.
    Set show_passes=True to see all.
    """
    print("\n" + "═" * 90)
    print(f"  {'RULE':<35} {'ROOM':<25} {'STATUS':<6} {'MEASURED':<15} {'REQUIRED'}")
    print("═" * 90)

    for r in results:
        if not show_passes and r.status == "PASS":
            continue
        icon = "✅" if r.status == "PASS" else "❌"
        label = f"{r.room_label[:20]} ({r.room_id})"
        print(
            f"  {icon} {r.rule_name[:33]:<33} {label:<25} {r.status:<6} "
            f"{r.measured[:14]:<15} {r.required}"
        )

    print("═" * 90)
    passed = sum(1 for r in results if r.status == "PASS")
    failed = sum(1 for r in results if r.status == "FAIL")
    total  = len(results)
    print(f"  SUMMARY:  {total} checks  |  {passed} PASSED  |  {failed} FAILED")
    print("═" * 90 + "\n")


# Print failures only
print_compliance_table(all_results, show_passes=False)

# Uncomment to see all results:
# print_compliance_table(all_results, show_passes=True)


══════════════════════════════════════════════════════════════════════════════════════════
  RULE                                ROOM                      STATUS MEASURED        REQUIRED
══════════════════════════════════════════════════════════════════════════════════════════
  ❌ Minimum room area                 TOILET (R000)             FAIL   0.47 m²         >= 2.0 m²
  ❌ Minimum room width                TOILET (R000)             FAIL   234 mm          >= 1200 mm
  ❌ Bedroom must have accessible toil CHILDREN BEDRM (R358)     FAIL   none found      adjacent to one of: TOILET, BATH, WC
  ❌ Bedroom must have accessible toil CHILDREN BEDRM (R720)     FAIL   none found      adjacent to one of: TOILET, BATH, WC
  ❌ Bedroom must have accessible toil CHILDREN BEDRM (R784)     FAIL   none found      adjacent to one of: TOILET, BATH, WC
  ❌ Bedroom must have accessible toil CHILDREN BEDRM (R862)     FAIL   none found      adjacent to one of: TOILET, BATH, WC
  ❌ Kitchen must connect to li

## Step 8: Build structured JSON context for the LLM

We send the LLM a **compact, structured summary** — not raw DXF or the full JSON.
This keeps the context tight and prevents hallucination.

In [74]:
def build_llm_context(rooms: list[Room], results: list[CheckResult]) -> str:
    """
    Builds the RAG context that goes into the LLM prompt.

    Contains:
    1. A compact room inventory (name, area, width)
    2. The deterministic check results (PASS/FAIL with measurements)
    3. Adjacency map (only for relevant rooms)

    The LLM's ONLY job is to:
    - Explain violations in plain English
    - Suggest fixes
    - Answer follow-up questions

    It does NOT re-measure or re-check — that was done in Python.
    """

    # Room inventory (labelled rooms only)
    room_inventory = []
    for r in rooms:
        if r.status == "unlabelled":
            continue
        room_inventory.append({
            "id":       r.id,
            "label":    r.raw_label,
            "area_m2":  r.area_m2,
            "width_mm": r.width_mm,
            "adjacent": [
                f"{room_index[adj].raw_label} ({adj})"
                for adj in r.adjacent
                if adj != "EXTERIOR" and adj in room_index
            ][:5],   # cap at 5 adjacents to keep context short
            "doors":    len(r.doors),
            "windows":  len(r.windows),
        })

    # Compliance results
    compliance_summary = [
        {
            "rule":     res.rule_name,
            "room":     res.room_label,
            "room_id":  res.room_id,
            "status":   res.status,
            "measured": res.measured,
            "required": res.required,
        }
        for res in results
    ]

    # Stats
    fails = [r for r in results if r.status == "FAIL"]
    stats = {
        "total_checks": len(results),
        "passed":        len(results) - len(fails),
        "failed":        len(fails),
        "total_rooms":   len([r for r in rooms if r.status != "unlabelled"]),
    }

    context = {
        "source":             "TOWER-B Ground Floor Plan",
        "statistics":         stats,
        "room_inventory":     room_inventory[:80],  # cap to avoid token overflow
        "compliance_results": compliance_summary,
    }

    return json.dumps(context, indent=2)


llm_context = build_llm_context(rooms, all_results)
print(f"✅ LLM context built: {len(llm_context):,} characters")

✅ LLM context built: 77,217 characters


## Step 9: Gemini LLM — explain violations and suggest fixes

In [81]:
import time
from google.genai.errors import ServerError

def query_gemini(context: str, question: str, api_key: str, model: str, max_retries: int = 4) -> str:
    client = genai.Client(api_key=api_key)

    system_prompt = """You are an architectural compliance expert..."""
    full_prompt = f"{system_prompt}\n\n=== DRAWING DATA AND COMPLIANCE RESULTS ===\n{context}\n\n=== QUESTION ===\n{question}"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=full_prompt
            )
            return response.text
        except ServerError as e:
            if attempt == max_retries - 1:
                print("❌ Max retries reached. Server is still busy.")
                raise e
            wait_time = 2 ** attempt  # 1s, 2s, 4s, 8s...
            print(f"⚠️ Model experiencing high demand (503). Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        except Exception as e:
            raise e

print("✅ Gemini query function ready with auto-retry.")

✅ Gemini query function ready with auto-retry.


## Step 10: Run the LLM analysis

In [84]:
# Default question — you can change this to anything about the drawing
question = """
1. List all compliance violations found, grouped by rule type.
2. For each violation, explain in one sentence why it matters architecturally.
3. For each violation, suggest one practical fix.
4. Summarise the overall compliance health of this floor plan.
"""

print("Querying Gemini... (this may take 10-20 seconds)\n")
llm_response = query_gemini(llm_context, question, GEMINI_API_KEY, GEMINI_MODEL)
print(llm_response)

Querying Gemini... (this may take 10-20 seconds)

### 1. Compliance Violations Grouped by Rule Type

Based on the architectural compliance audit for **TOWER-B Ground Floor Plan** (total 266 checks: 255 passed, 11 failed across 189 rooms), the following 11 violations were detected:

#### **A. Minimum Room Area & Width**
*   **TOILET (R000)**
    *   *Minimum room area:* Measured `0.47 m²` (Required $\ge 2.0\text{ m}^2$) — **FAIL**
    *   *Minimum room width:* Measured `234 mm` (Required $\ge 1200\text{ mm}$) — **FAIL**

#### **B. Bedroom Must Have Accessible Toilet**
*   **CHILDREN BEDRM (R358)**
    *   *Rule Check:* Measured `none found` (Required: adjacent to TOILET, BATH, or WC) — **FAIL**
*   **CHILDREN BEDRM (R720)**
    *   *Rule Check:* Measured `none found` (Required: adjacent to TOILET, BATH, or WC) — **FAIL**
*   **CHILDREN BEDRM (R784)**
    *   *Rule Check:* Measured `none found` (Required: adjacent to TOILET, BATH, WC) — **FAIL**
*   **CHILDREN BEDRM (R862)**
    *   *Rul

## Step 11: Interactive Q&A — ask anything about the drawing

In [85]:
# Change this question and re-run the cell to ask anything
custom_question = "Which rooms fail the minimum area check and by how much?"

response = query_gemini(llm_context, custom_question, GEMINI_API_KEY, GEMINI_MODEL)
print(response)

Based on the architectural compliance results provided, only **one** room fails the minimum area check:

* **Room:** TOILET (`R000`)
* **Measured Area:** 0.47 m²
* **Required Area:** ≥ 2.0 m²
* **Deficit (Fails by):** **1.53 m²** (or 76.5% below the minimum requirement)


In [86]:
# Another example
custom_question = "Are all bedrooms properly connected to a toilet or bathroom?"

response = query_gemini(llm_context, custom_question, GEMINI_API_KEY, GEMINI_MODEL)
print(response)

Based on the compliance check results, **no, not all bedrooms are properly connected to a toilet or bathroom.** 

While the majority of bedrooms comply with the rule (requiring adjacency to a toilet, bath, or WC), there are **4 bedrooms that failed** this compliance check because no accessible toilet was found adjacent to them:

1. **CHILDREN BEDRM (R358)**
2. **CHILDREN BEDRM (R720)**
3. **CHILDREN BEDRM (R784)**
4. **CHILDREN BEDRM (R862)**


## Step 12: Export compliance report to JSON

In [87]:
def export_report(results: list[CheckResult], output_path: str = "compliance_report.json"):
    """
    Saves the full compliance report as a JSON file.
    This can feed into your existing frontend dashboard.
    """
    report = {
        "summary": {
            "total":  len(results),
            "passed": sum(1 for r in results if r.status == "PASS"),
            "failed": sum(1 for r in results if r.status == "FAIL"),
        },
        "violations": [
            {
                "rule":     r.rule_name,
                "room_id":  r.room_id,
                "room":     r.room_label,
                "status":   r.status,
                "measured": r.measured,
                "required": r.required,
                "detail":   r.detail,
            }
            for r in results
            if r.status == "FAIL"
        ],
        "all_checks": [
            {
                "rule":     r.rule_name,
                "room_id":  r.room_id,
                "room":     r.room_label,
                "status":   r.status,
                "measured": r.measured,
                "required": r.required,
            }
            for r in results
        ],
    }

    with open(output_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"✅ Report saved to {output_path}")
    print(f"   {report['summary']['failed']} violations out of {report['summary']['total']} checks")
    return report


report = export_report(all_results)

✅ Report saved to compliance_report.json
   11 violations out of 266 checks


---

## How to extend this pipeline

**Add a new rule:**
```python
# In RULES dict:
"WAITING":  (10.0, 2400),   # adds area + width check for waiting rooms
```

**Add a new adjacency rule:**
```python
# In ADJACENCY_RULES list:
{
    "rule_name":    "Lift must be near staircase",
    "source_types": ["LIFT"],
    "must_be_near": ["STAIRCASE", "FIRE TOWER"],
    "description":  "Lifts must be adjacent to emergency stairs",
}
```

**Ask a specific question to Gemini:**
```python
custom_question = "Which units fail for master bedroom size?"
response = query_gemini(llm_context, custom_question, GEMINI_API_KEY, GEMINI_MODEL)
print(response)
```

**Use with Apollo hospital drawings:**
Just change `MANIFEST_PATH` and update `RULES` with NBC hospital-specific thresholds
(ICU >= 10 sqm, OT >= 36 sqm, corridors >= 2400mm, etc.).
